# Workshop 10: Decision Trees — Interpretable Machine Learning

---

## What are Decision Trees?

A **Decision Tree** is a supervised learning algorithm that predicts the value of a target variable by learning simple **if-then-else decision rules** inferred from the data features. The model can be visualised as a tree structure where:

- Each **internal node** represents a test on a feature (e.g., `alcohol > 12.5`)
- Each **branch** represents the outcome of a test
- Each **leaf node** represents a class label (classification) or a predicted value (regression)

Decision trees are used in a wide variety of domains:
- **Medical diagnosis** (is this tumour malignant?)
- **Credit scoring** (will this customer default?)
- **Fraud detection** (is this transaction suspicious?)
- **Customer segmentation** (which product will this user buy?)

## Why Interpretability Matters

Many high-stakes applications require models that can explain their decisions:

| Domain | Requirement |
|--------|-------------|
| **Healthcare** | Clinicians need to understand *why* a diagnosis was made |
| **Finance** | Regulations (e.g., GDPR, Fair Lending Act) require explanations for loan denials |
| **Law** | Algorithmic decisions in courts must be auditable |
| **Engineering** | Fault diagnosis trees are used in safety-critical systems |

Decision trees offer **white-box** interpretability: you can follow the path from root to leaf and understand every decision made.

## Learning Objectives

By the end of this workshop, you will be able to:

1. **Explain** the recursive binary splitting algorithm and how impurity measures guide it
2. **Compute** Gini impurity, entropy, and information gain by hand and in Python
3. **Train, visualise, and interpret** a decision tree using scikit-learn
4. **Diagnose and control** overfitting through hyperparameter tuning
5. **Compare** classification and regression trees and understand when to use each

## Theory: How Decision Trees Work

### Recursive Binary Splitting

Building a decision tree is a **greedy, top-down, recursive** procedure:

1. Start with all training samples at the root node
2. Find the **best feature** and the **best threshold** to split on (maximising some criterion)
3. Partition the data into two child nodes based on the split
4. Repeat recursively for each child node until a stopping criterion is met

The algorithm is *greedy* because at each step it makes the locally optimal split — it does not backtrack to reconsider earlier choices.

### Nodes, Branches, and Leaves

```
                    [Root Node]
                   alcohol > 12.5?
                   /             \
               YES /               \ NO
                  /                 \
         [Internal Node]       [Internal Node]
        flavanoids > 2.1?     malic_acid > 2.4?
           /       \               /       \
          /         \             /         \
      [Leaf]      [Leaf]       [Leaf]      [Leaf]
     Class 0     Class 1      Class 2     Class 1
```

- **Root Node**: the topmost node, contains all samples
- **Internal Node**: a node that has been split; contains a decision rule
- **Branch / Edge**: the connection between a parent and a child node
- **Leaf Node** (terminal node): no further splits; holds the final prediction

### The Key Question: How Do We Choose the Best Split?

At each node, we must choose:
- **Which feature** $j$ to split on
- **Which threshold** $t$ to use for that feature

We evaluate every possible $(j, t)$ pair and pick the one that results in the **greatest reduction in impurity** of the child nodes relative to the parent node.

The formal criterion is defined in the next section.

## Theory: Impurity Measures

An **impurity measure** quantifies how mixed the class labels are in a node. A pure node (all samples belong to the same class) has impurity = 0.

Let $p_k$ be the proportion of samples belonging to class $k$ at node $t$, with $K$ total classes.

---

### 1. Gini Impurity

$$\text{Gini}(t) = 1 - \sum_{k=1}^{K} p_k^2$$

- Ranges from $0$ (pure) to $1 - \frac{1}{K}$ (maximally impure)
- For binary classification: $\text{Gini}(t) = 1 - p_0^2 - p_1^2 = 2 p_0 (1 - p_0)$
- Maximum value for binary case: $0.5$ (when $p_0 = p_1 = 0.5$)

---

### 2. Entropy (Information-Theoretic)

$$H(t) = -\sum_{k=1}^{K} p_k \log_2(p_k)$$

where by convention $0 \cdot \log_2(0) = 0$.

- Ranges from $0$ (pure) to $\log_2(K)$ (maximally impure)
- For binary classification: maximum entropy is $1$ bit (when $p_0 = p_1 = 0.5$)

---

### 3. Information Gain

The **Information Gain** of a split that divides node $t$ (with $n$ samples) into left child $t_L$ ($n_L$ samples) and right child $t_R$ ($n_R$ samples) is:

$$IG(t, \text{split}) = H(t) - \left(\frac{n_L}{n} \cdot H(t_L) + \frac{n_R}{n} \cdot H(t_R)\right)$$

The weighted average of children entropies is called the **conditional entropy**. We maximise $IG$ to find the best split.

---

### Worked Numeric Example

Suppose we have a node with 10 samples: **6 of class A** and **4 of class B**.

**Parent node impurity:**

$$p_A = 0.6, \quad p_B = 0.4$$

$$H(\text{parent}) = -(0.6 \log_2 0.6 + 0.4 \log_2 0.4) = -(0.6 \times (-0.737) + 0.4 \times (-1.322)) \approx 0.971 \text{ bits}$$

$$\text{Gini}(\text{parent}) = 1 - (0.6^2 + 0.4^2) = 1 - (0.36 + 0.16) = 0.48$$

Now consider **Split A**: Left = {4A, 1B}, Right = {2A, 3B}

$$H(t_L) = -(0.8 \log_2 0.8 + 0.2 \log_2 0.2) \approx 0.722 \text{ bits}$$

$$H(t_R) = -(0.4 \log_2 0.4 + 0.6 \log_2 0.6) \approx 0.971 \text{ bits}$$

$$IG_A = 0.971 - \left(\frac{5}{10} \times 0.722 + \frac{5}{10} \times 0.971\right) = 0.971 - 0.847 = 0.124 \text{ bits}$$

Now consider **Split B**: Left = {6A, 0B}, Right = {0A, 4B} ← *perfectly pure!*

$$H(t_L) = 0, \quad H(t_R) = 0$$

$$IG_B = 0.971 - \left(\frac{6}{10} \times 0 + \frac{4}{10} \times 0\right) = 0.971 \text{ bits}$$

Split B is clearly better!

---

### Gini vs Entropy: When Do They Differ?

| Property | Gini Impurity | Entropy |
|----------|--------------|----------|
| **Range (binary)** | $[0, 0.5]$ | $[0, 1]$ |
| **Computation** | Faster (no log) | Slightly slower |
| **Sensitivity** | Moderate | More sensitive to changes near $p=0.5$ |
| **Splits** | Tends to isolate the most frequent class | Tends toward more balanced splits |

In practice, **both metrics yield very similar trees**. Gini is the scikit-learn default for speed. Use entropy when you want slightly more balanced splits or when information-theoretic interpretation matters.

## Theory: Tree Building Algorithm — CART

### The CART Algorithm

Scikit-learn implements the **CART** (Classification and Regression Trees) algorithm, introduced by Breiman et al. (1984). CART produces **binary trees** only (each node has exactly two children).

**Algorithm (Classification):**

```
function BuildTree(node, X, y):
    if StoppingCriterion(node, X, y):
        node.prediction = majority_class(y)
        return
    
    best_feature, best_threshold = FindBestSplit(X, y)
    node.rule = (best_feature, best_threshold)
    
    left_mask  = X[:, best_feature] <= best_threshold
    right_mask = ~left_mask
    
    BuildTree(node.left,  X[left_mask],  y[left_mask])
    BuildTree(node.right, X[right_mask], y[right_mask])
```

### Stopping Criteria

Without stopping criteria, CART will grow until every leaf is pure — which leads to massive overfitting. The main stopping criteria in scikit-learn are:

| Parameter | Description | Effect |
|-----------|-------------|--------|
| `max_depth` | Maximum depth of the tree | Directly limits model complexity |
| `min_samples_split` | Min. samples required to split a node | Prevents splitting very small nodes |
| `min_samples_leaf` | Min. samples required at a leaf node | Ensures leaves have enough support |
| `min_impurity_decrease` | Min. impurity reduction required to split | Only splits if gain is "worth it" |
| `max_leaf_nodes` | Maximum number of leaf nodes | Alternative way to limit size |

### Pruning: Pre-pruning vs Post-pruning

**Pre-pruning** (Early Stopping): Stop growing the tree *during construction* using the stopping criteria above. Simple and fast, but may stop too early.

**Post-pruning**: Grow the full tree first, then *remove* subtrees that do not improve generalisation on a validation set.

Scikit-learn implements **Cost-Complexity Pruning** (also called weakest link pruning) via the `ccp_alpha` parameter:

$$R_\alpha(T) = R(T) + \alpha \cdot |T|$$

where $R(T)$ is the total misclassification rate of the tree $T$, $|T|$ is the number of leaf nodes, and $\alpha \geq 0$ is the complexity parameter. Larger $\alpha$ penalises complexity more, resulting in smaller trees.

## Theory: Overfitting in Decision Trees

### The Overfitting Problem

A fully grown decision tree (no stopping criteria) will **perfectly classify every training sample** — each leaf will contain only samples from one class, giving 0 training error. But this comes at a severe cost to generalisation.

Consider what happens:
- With 100 training samples and 10 features, a deep tree can create regions so specific that they contain just 1–2 samples
- These regions capture **noise** in the training data, not true patterns
- On unseen data, these overly specific rules fail

### Bias-Variance Tradeoff

The generalisation error of any model can be decomposed as:

$$\text{Generalisation Error} = \text{Bias}^2 + \text{Variance} + \text{Noise}$$

- **Bias**: Error from wrong assumptions about the data structure (underfitting)
- **Variance**: Error from sensitivity to small fluctuations in the training set (overfitting)
- **Noise**: Irreducible error inherent in the data

For decision trees:

| Tree Depth | Bias | Variance | Result |
|-----------|------|----------|--------|
| Very shallow (depth=1) | High | Low | Underfitting — too simple |
| Moderate (depth=3–7) | Low-Moderate | Low-Moderate | Good generalisation |
| Unlimited | Very Low | Very High | Overfitting — memorises training data |

### How Depth Controls Complexity

The decision boundary of a tree is defined by a set of **axis-aligned rectangular regions** in feature space. As depth increases:
- More rectangles are created
- The boundaries become more jagged and specific
- The model captures increasingly fine-grained patterns (eventually: just noise)

The optimal depth is typically found by **cross-validation** on a held-out set.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine, make_regression
from sklearn.model_selection import train_test_split, cross_val_score, validation_curve
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("All libraries imported successfully!")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Implementing Gini Impurity, Entropy, and Information Gain
# from scratch — no scikit-learn here!
# ─────────────────────────────────────────────────────────────

def gini_impurity(y):
    """
    Compute Gini impurity for an array of class labels.
    Gini(t) = 1 - sum(p_k^2)
    """
    if len(y) == 0:
        return 0.0
    classes, counts = np.unique(y, return_counts=True)
    proportions = counts / len(y)
    return 1.0 - np.sum(proportions ** 2)


def entropy(y):
    """
    Compute Shannon entropy for an array of class labels.
    H(t) = -sum(p_k * log2(p_k))
    """
    if len(y) == 0:
        return 0.0
    classes, counts = np.unique(y, return_counts=True)
    proportions = counts / len(y)
    # Avoid log(0) by filtering zeros
    proportions = proportions[proportions > 0]
    return -np.sum(proportions * np.log2(proportions))


def information_gain(y_parent, y_left, y_right, criterion='entropy'):
    """
    Compute Information Gain of a binary split.

    Parameters
    ----------
    y_parent : array-like  — labels at the parent node
    y_left   : array-like  — labels going to the left child
    y_right  : array-like  — labels going to the right child
    criterion: 'entropy' or 'gini'

    Returns
    -------
    float : information gain (higher is better)
    """
    impurity_fn = entropy if criterion == 'entropy' else gini_impurity
    n = len(y_parent)
    n_L, n_R = len(y_left), len(y_right)

    parent_impurity = impurity_fn(y_parent)
    weighted_children = (n_L / n) * impurity_fn(y_left) + (n_R / n) * impurity_fn(y_right)
    return parent_impurity - weighted_children


# ─── Demo with a small manual example ───
print("=" * 55)
print("DEMO: Impurity Measures on a Toy Dataset")
print("=" * 55)

# 10 samples: 6 class A (0), 4 class B (1)
y_parent = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1])

print(f"\nParent node: {dict(zip(*np.unique(y_parent, return_counts=True)))}")
print(f"  Gini impurity : {gini_impurity(y_parent):.4f}")
print(f"  Entropy       : {entropy(y_parent):.4f} bits")

# Split A: Left = [4A, 1B], Right = [2A, 3B]
y_left_A  = np.array([0, 0, 0, 0, 1])
y_right_A = np.array([0, 0, 1, 1, 1])

print(f"\nSplit A — Left: {{0:4, 1:1}}, Right: {{0:2, 1:3}}")
print(f"  IG (entropy)  : {information_gain(y_parent, y_left_A, y_right_A, 'entropy'):.4f} bits")
print(f"  IG (gini)     : {information_gain(y_parent, y_left_A, y_right_A, 'gini'):.4f}")

# Split B: Left = [6A, 0B], Right = [0A, 4B] — perfect split!
y_left_B  = np.array([0, 0, 0, 0, 0, 0])
y_right_B = np.array([1, 1, 1, 1])

print(f"\nSplit B — Left: {{0:6, 1:0}}, Right: {{0:0, 1:4}} (perfect!)")
print(f"  IG (entropy)  : {information_gain(y_parent, y_left_B, y_right_B, 'entropy'):.4f} bits")
print(f"  IG (gini)     : {information_gain(y_parent, y_left_B, y_right_B, 'gini'):.4f}")

# Visual comparison
print("\nConclusion: Split B has higher IG → CART will prefer Split B")

# ─── Plot: Gini vs Entropy as a function of p (binary case) ───
p_values = np.linspace(0.001, 0.999, 200)
gini_vals    = 2 * p_values * (1 - p_values)
entropy_vals = -(p_values * np.log2(p_values) + (1 - p_values) * np.log2(1 - p_values))
misclass_vals = 1 - np.maximum(p_values, 1 - p_values)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p_values, gini_vals,    label='Gini (×2 for scale)',   color='steelblue', lw=2)
ax.plot(p_values, entropy_vals, label='Entropy (bits)',         color='coral',     lw=2)
ax.plot(p_values, misclass_vals,label='Misclassification rate', color='green',     lw=2, ls='--')
ax.set_xlabel('Proportion of class 1  ($p$)', fontsize=12)
ax.set_ylabel('Impurity', fontsize=12)
ax.set_title('Impurity Measures for Binary Classification', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.axvline(0.5, color='gray', ls=':', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Load the Wine Dataset & Exploratory Data Analysis
# ─────────────────────────────────────────────────────────────

wine = load_wine()
X    = wine.data
y    = wine.target
feature_names = wine.feature_names
class_names   = wine.target_names

# Create a DataFrame for EDA
df = pd.DataFrame(X, columns=feature_names)
df['class'] = y
df['class_name'] = [class_names[i] for i in y]

print("=" * 55)
print("Wine Dataset Overview")
print("=" * 55)
print(f"Shape            : {X.shape}  ({X.shape[0]} samples, {X.shape[1]} features)")
print(f"Number of classes: {len(class_names)}")
print(f"Class names      : {list(class_names)}")
print(f"\nClass distribution:")
for i, name in enumerate(class_names):
    count = np.sum(y == i)
    print(f"  Class {i} ({name}): {count} samples ({100*count/len(y):.1f}%)")

print("\nFirst 5 rows:")
print(df.head())

print("\nFeature statistics (mean per class):")
print(df.groupby('class_name')[feature_names].mean().round(2).T)

# ─── Plot 1: Class distribution ───
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

class_counts = pd.Series(y).value_counts().sort_index()
colors = ['#4C72B0', '#DD8452', '#55A868']
axes[0].bar([f'Class {i}\n({class_names[i]})' for i in range(3)],
            class_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# ─── Plot 2: Feature correlation heatmap ───
corr_with_target = df[feature_names].corrwith(pd.Series(y.astype(float))).abs().sort_values(ascending=False)
top4 = corr_with_target.index[:4].tolist()
sns.heatmap(df[top4 + ['class']].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=axes[1], linewidths=0.5)
axes[1].set_title('Correlation Matrix (Top 4 Features + Class)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ─── Plot 3: Pairplot of top 4 correlated features ───
print(f"\nTop 4 features correlated with class: {top4}")
pair_df = df[top4 + ['class']].copy()
pair_df['class'] = pair_df['class'].astype(str)

g = sns.pairplot(pair_df, hue='class', palette={"0": colors[0], "1": colors[1], "2": colors[2]},
                 plot_kws={'alpha': 0.6, 's': 40}, diag_kind='kde')
g.fig.suptitle('Pairplot of Top 4 Most Discriminative Features', y=1.02, fontsize=13, fontweight='bold')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Preprocessing: Train / Test Split
# ─────────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,       # preserve class proportions
    random_state=42
)

print("Train / Test Split (80 / 20, stratified)")
print("=" * 45)
print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")

print("\nClass balance in training set:")
for i, name in enumerate(class_names):
    n = np.sum(y_train == i)
    print(f"  Class {i} ({name}): {n} ({100*n/len(y_train):.1f}%)")

print("\nClass balance in test set:")
for i, name in enumerate(class_names):
    n = np.sum(y_test == i)
    print(f"  Class {i} ({name}): {n} ({100*n/len(y_test):.1f}%)")

print("\nNote: Decision trees do NOT require feature scaling!")
print("  → Trees make decisions based on thresholds (e.g., alcohol > 12.5)")
print("  → Scaling a feature does not change the ranking of thresholds")
print("  → Unlike KNN or SVM, trees are invariant to monotonic feature transforms")

## Building Our First Decision Tree

We will start with a **shallow tree** (`max_depth=3`). This is a deliberate pedagogical choice:

- A depth-3 tree has at most $2^3 = 8$ leaves — small enough to visualise and read
- Each path from root to leaf encodes a **human-readable rule** (at most 3 conditions)
- Shallow trees are less likely to overfit, making them a solid first baseline

We can then increase depth incrementally and observe the effect on train/test accuracy. This will demonstrate the bias-variance tradeoff we discussed in the theory section.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Train Decision Tree with max_depth=3
# ─────────────────────────────────────────────────────────────

clf = DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    random_state=42
)

clf.fit(X_train, y_train)

y_pred_train = clf.predict(X_train)
y_pred_test  = clf.predict(X_test)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc  = accuracy_score(y_test,  y_pred_test)

print("Decision Tree (max_depth=3, criterion='gini')")
print("=" * 45)
print(f"Training accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)")
print(f"Test accuracy     : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"\nTree depth (actual)  : {clf.get_depth()}")
print(f"Number of leaves     : {clf.get_n_leaves()}")
print(f"Number of nodes      : {clf.tree_.node_count}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Visualisation 1: Plot the Decision Tree
# ─────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(20, 10))

plot_tree(
    clf,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,          # colour nodes by majority class
    rounded=True,         # rounded boxes
    fontsize=10,
    impurity=True,        # show Gini impurity
    proportion=False,     # show raw sample counts (not proportions)
    ax=ax
)

ax.set_title(
    'Decision Tree on Wine Dataset  (max_depth=3, criterion=Gini)',
    fontsize=14, fontweight='bold', pad=15
)

plt.tight_layout()
plt.show()

print("\nHow to read this tree:")
print("  • Each node shows: feature threshold | Gini impurity | # samples | class counts")
print("  • Colour intensity reflects purity (darker = more dominant class)")
print("  • Follow TRUE branch (left) or FALSE branch (right) at each node")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Visualisation 2: Text Representation of the Tree
# ─────────────────────────────────────────────────────────────

tree_text = export_text(clf, feature_names=list(feature_names))

print("Decision Tree as Text Rules")
print("=" * 55)
print(tree_text)

print("How to read the text representation:")
print("  |--- feature <= threshold    → left branch (condition TRUE)")
print("  |--- feature >  threshold    → right branch (condition FALSE)")
print("  |   |--- ...                 → deeper subtree")
print("  |--- class: X               → leaf node, predicts class X")
print("\nExample rule path for Class 0 (leftmost leaf):")
print("  IF proline <= 755.0")
print("  AND (follow the left subtree until a leaf is reached)")
print("  THEN predict class_name[predicted_class]")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Feature Importance — Bar Chart
# ─────────────────────────────────────────────────────────────

importances = clf.feature_importances_
indices = np.argsort(importances)  # sort ascending for horizontal bar

fig, ax = plt.subplots(figsize=(9, 7))

bars = ax.barh(
    range(len(indices)),
    importances[indices],
    color=plt.cm.viridis(np.linspace(0.2, 0.85, len(indices))),
    edgecolor='white', linewidth=0.8
)

ax.set_yticks(range(len(indices)))
ax.set_yticklabels([feature_names[i] for i in indices], fontsize=10)
ax.set_xlabel('Feature Importance (Mean Decrease in Gini Impurity)', fontsize=11)
ax.set_title('Feature Importances — Decision Tree (max_depth=3)', fontsize=13, fontweight='bold')

# Annotate bar values
for i, (bar, val) in enumerate(zip(bars, importances[indices])):
    if val > 0:
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=9, color='#333333')

ax.set_xlim(0, importances.max() + 0.08)
plt.tight_layout()
plt.show()

print("\nTop 3 most important features:")
top_indices = np.argsort(importances)[::-1][:3]
for rank, idx in enumerate(top_indices, 1):
    print(f"  {rank}. {feature_names[idx]:<30s} importance = {importances[idx]:.4f}")

## Theory: Feature Importance in Decision Trees

### How Scikit-learn Computes Feature Importance

The feature importance of feature $j$ is defined as the **total (normalised) reduction in impurity** brought by splits on that feature across the entire tree:

$$\text{Importance}(j) = \frac{1}{N} \sum_{t : \text{split on feature } j} n_t \cdot IG(t)$$

where $n_t$ is the number of samples reaching node $t$ and $N$ is the total number of training samples. The importances are then normalised so they sum to 1.

This is also called **Mean Decrease in Impurity (MDI)**.

### Limitations of MDI

MDI has a well-known bias:

- **High-cardinality features** (many unique values) can be split at more thresholds, giving them more opportunities to reduce impurity → artificially inflated importance
- Example: a random integer ID column might appear important simply because it creates perfectly pure splits by memorising samples
- This bias is especially pronounced for **random forests** on mixed feature types

### Permutation Importance — A More Robust Alternative

**Permutation Importance** (Breiman, 2001) measures how much model performance decreases when a single feature is randomly shuffled:

$$\text{PI}(j) = \text{score}(\text{original}) - \text{score}(\text{feature } j \text{ shuffled})$$

- Does not depend on the model internals — works as a black-box evaluation
- Less biased toward high-cardinality features
- Computationally more expensive (requires multiple model evaluations)
- Available in scikit-learn via `sklearn.inspection.permutation_importance`

**Recommendation**: Use MDI for a quick sanity check, permutation importance for reliable feature selection.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Overfitting Demonstration: Accuracy vs Tree Depth
# ─────────────────────────────────────────────────────────────

depths = range(1, 16)
train_scores = []
test_scores  = []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, tree.predict(X_train)))
    test_scores.append(accuracy_score(y_test,  tree.predict(X_test)))

best_depth = depths[np.argmax(test_scores)]
best_score = max(test_scores)

print(f"Best test accuracy: {best_score:.4f} at max_depth={best_depth}")

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(depths, train_scores, 'o-', color='steelblue', label='Train Accuracy', lw=2, ms=6)
ax.plot(depths, test_scores,  's-', color='coral',     label='Test Accuracy',  lw=2, ms=6)

# Highlight best
ax.axvline(best_depth, color='green', ls='--', alpha=0.7, label=f'Sweet spot (depth={best_depth})')
ax.annotate(
    f'Best test\n{best_score:.2%}',
    xy=(best_depth, best_score),
    xytext=(best_depth + 1, best_score - 0.04),
    arrowprops=dict(arrowstyle='->', color='green'),
    fontsize=10, color='green'
)

# Shade overfitting zone
ax.axvspan(best_depth + 0.5, 15.5, alpha=0.08, color='red', label='Overfitting zone')

ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Overfitting in Decision Trees: Train vs Test Accuracy', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0.5, 1.05)
ax.set_xticks(list(depths))
plt.tight_layout()
plt.show()

print("\nObservations:")
print("  • Train accuracy reaches 1.0 as depth increases — perfect memorisation")
print("  • Test accuracy peaks then decreases or plateaus — generalisation limit")
print(f"  • Gap between train and test grows after depth={best_depth} → overfitting")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Validation Curve using sklearn's validation_curve
# (Cross-validated — more reliable than a single split)
# ─────────────────────────────────────────────────────────────

param_range = np.arange(1, 21)

train_scores_vc, val_scores_vc = validation_curve(
    DecisionTreeClassifier(random_state=42),
    X, y,
    param_name='max_depth',
    param_range=param_range,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

train_mean = train_scores_vc.mean(axis=1)
train_std  = train_scores_vc.std(axis=1)
val_mean   = val_scores_vc.mean(axis=1)
val_std    = val_scores_vc.std(axis=1)

best_depth_cv = param_range[np.argmax(val_mean)]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(param_range, train_mean, 'o-', color='steelblue', label='Training score', lw=2)
ax.fill_between(param_range,
                train_mean - train_std,
                train_mean + train_std,
                alpha=0.15, color='steelblue')

ax.plot(param_range, val_mean, 's-', color='coral', label='Cross-validation score', lw=2)
ax.fill_between(param_range,
                val_mean - val_std,
                val_mean + val_std,
                alpha=0.15, color='coral')

ax.axvline(best_depth_cv, color='green', ls='--', alpha=0.8,
           label=f'Optimal depth = {best_depth_cv}  (CV acc = {val_mean[best_depth_cv-1]:.3f})')

ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Validation Curve for max_depth  (5-fold CV, Wine Dataset)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0.5, 1.05)
ax.set_xticks(param_range)
plt.tight_layout()
plt.show()

print(f"Optimal max_depth (by CV): {best_depth_cv}")
print(f"CV accuracy at optimal depth: {val_mean[best_depth_cv-1]:.4f} ± {val_std[best_depth_cv-1]:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Confusion Matrix — Best Model
# ─────────────────────────────────────────────────────────────

# Train with the CV-optimal depth
clf_best = DecisionTreeClassifier(max_depth=best_depth_cv, random_state=42)
clf_best.fit(X_train, y_train)
y_pred_best = clf_best.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(7, 6))

sns.heatmap(
    cm,
    annot=True, fmt='d',
    cmap='Blues',
    xticklabels=[f'Pred\n{n}' for n in class_names],
    yticklabels=[f'True\n{n}' for n in class_names],
    linewidths=0.8, linecolor='white',
    annot_kws={'size': 14, 'weight': 'bold'},
    ax=ax
)

ax.set_title(
    f'Confusion Matrix — Decision Tree (max_depth={best_depth_cv})\n'
    f'Test Accuracy: {accuracy_score(y_test, y_pred_best):.2%}',
    fontsize=12, fontweight='bold'
)
ax.set_ylabel('True Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nTest accuracy (best model, depth={best_depth_cv}): {accuracy_score(y_test, y_pred_best):.4f}")
print(f"\nConfusion matrix:")
print(cm)
print("\nDiagonal = correct predictions, off-diagonal = errors")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Full Evaluation: Classification Report + Cross-Validation
# ─────────────────────────────────────────────────────────────

print("Classification Report — Best Model (max_depth={})\n".format(best_depth_cv))
print(classification_report(y_test, y_pred_best, target_names=class_names))

# 5-fold cross-validation
cv_scores_best = cross_val_score(
    DecisionTreeClassifier(max_depth=best_depth_cv, random_state=42),
    X, y, cv=5, scoring='accuracy'
)

print("5-fold Cross-Validation Accuracy (max_depth={})".format(best_depth_cv))
print(f"  Fold scores : {np.round(cv_scores_best, 4)}")
print(f"  Mean ± Std  : {cv_scores_best.mean():.4f} ± {cv_scores_best.std():.4f}")

# Comparison with a fully grown tree (no depth limit)
clf_full = DecisionTreeClassifier(random_state=42)   # no max_depth
clf_full.fit(X_train, y_train)
y_pred_full = clf_full.predict(X_test)

cv_scores_full = cross_val_score(
    DecisionTreeClassifier(random_state=42),
    X, y, cv=5, scoring='accuracy'
)

print("\n" + "=" * 55)
print("Comparison: Pruned vs Fully Grown Tree")
print("=" * 55)
print(f"{'Model':<30} {'Train Acc':>10} {'Test Acc':>10} {'CV Mean':>10} {'# Leaves':>10}")
print("-" * 72)
print(f"{'Pruned (max_depth=' + str(best_depth_cv) + ')':<30}"
      f" {accuracy_score(y_train, clf_best.predict(X_train)):>10.4f}"
      f" {accuracy_score(y_test, y_pred_best):>10.4f}"
      f" {cv_scores_best.mean():>10.4f}"
      f" {clf_best.get_n_leaves():>10}")
print(f"{'Fully Grown (no limit)':<30}"
      f" {accuracy_score(y_train, clf_full.predict(X_train)):>10.4f}"
      f" {accuracy_score(y_test, y_pred_full):>10.4f}"
      f" {cv_scores_full.mean():>10.4f}"
      f" {clf_full.get_n_leaves():>10}")

## Theory: Hyperparameters Deep Dive

Decision trees have several key hyperparameters that control the **complexity** of the learned model. Choosing them wisely is crucial to avoid both underfitting and overfitting.

| Hyperparameter | What it Controls | Recommended Starting Value |
|---------------|-----------------|---------------------------|
| `max_depth` | Maximum depth of the tree; directly caps model complexity | 3–10; tune with CV |
| `min_samples_split` | Minimum number of samples a node must have to be split further | 2 (default); try 5–20 for noisy data |
| `min_samples_leaf` | Minimum number of samples that must be in a leaf after a split | 1 (default); try 2–10; smooths predictions |
| `max_features` | Number of features to consider at each split; introduces randomness | `None` (all); `"sqrt"` for Random Forests |
| `criterion` | Impurity measure used to evaluate splits | `'gini'` (default) or `'entropy'`; rarely matters much |
| `max_leaf_nodes` | Maximum number of leaf nodes; alternative to `max_depth` | `None` (unlimited); use when you want to cap model size explicitly |
| `ccp_alpha` | Complexity parameter for cost-complexity pruning (post-pruning) | 0.0 (no pruning); tune by plotting accuracy vs alpha |

### General Tuning Strategy

1. **Start simple**: Try `max_depth=3` or `max_depth=5` first
2. **Use cross-validation**: Never rely on a single train/test split to select hyperparameters
3. **Prioritise interpretability**: If the business requires explainability, keep `max_depth ≤ 5`
4. **Consider pruning**: `ccp_alpha` often gives a better accuracy-complexity tradeoff than `max_depth` alone
5. **Combine constraints**: `max_depth=5` + `min_samples_leaf=5` together prevent very fine-grained splits

In [ ]:
# ─────────────────────────────────────────────────────────────
# Hyperparameter Effect Visualisation
# 4 subplots, each showing one hyperparameter's effect
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# ── 1. max_depth ──
depths = range(1, 16)
tr, te = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    tr.append(accuracy_score(y_train, m.predict(X_train)))
    te.append(accuracy_score(y_test,  m.predict(X_test)))
axes[0].plot(depths, tr, 'o-', color='steelblue', label='Train', lw=2)
axes[0].plot(depths, te, 's-', color='coral',     label='Test',  lw=2)
axes[0].set_title('Effect of max_depth', fontweight='bold')
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].set_ylim(0.5, 1.05)

# ── 2. min_samples_split ──
mss_vals = range(2, 30)
tr, te = [], []
for v in mss_vals:
    m = DecisionTreeClassifier(min_samples_split=v, random_state=42).fit(X_train, y_train)
    tr.append(accuracy_score(y_train, m.predict(X_train)))
    te.append(accuracy_score(y_test,  m.predict(X_test)))
axes[1].plot(mss_vals, tr, 'o-', color='steelblue', label='Train', lw=2)
axes[1].plot(mss_vals, te, 's-', color='coral',     label='Test',  lw=2)
axes[1].set_title('Effect of min_samples_split', fontweight='bold')
axes[1].set_xlabel('min_samples_split'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].set_ylim(0.5, 1.05)

# ── 3. min_samples_leaf ──
msl_vals = range(1, 25)
tr, te = [], []
for v in msl_vals:
    m = DecisionTreeClassifier(min_samples_leaf=v, random_state=42).fit(X_train, y_train)
    tr.append(accuracy_score(y_train, m.predict(X_train)))
    te.append(accuracy_score(y_test,  m.predict(X_test)))
axes[2].plot(msl_vals, tr, 'o-', color='steelblue', label='Train', lw=2)
axes[2].plot(msl_vals, te, 's-', color='coral',     label='Test',  lw=2)
axes[2].set_title('Effect of min_samples_leaf', fontweight='bold')
axes[2].set_xlabel('min_samples_leaf'); axes[2].set_ylabel('Accuracy')
axes[2].legend(); axes[2].set_ylim(0.5, 1.05)

# ── 4. max_features ──
mf_vals   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
mf_labels = [str(v) for v in mf_vals]
tr, te = [], []
for v in mf_vals:
    m = DecisionTreeClassifier(max_features=v, random_state=42).fit(X_train, y_train)
    tr.append(accuracy_score(y_train, m.predict(X_train)))
    te.append(accuracy_score(y_test,  m.predict(X_test)))
axes[3].plot(mf_vals, tr, 'o-', color='steelblue', label='Train', lw=2)
axes[3].plot(mf_vals, te, 's-', color='coral',     label='Test',  lw=2)
axes[3].axvline(13, color='gray', ls=':', label='All features (13)')
axes[3].set_title('Effect of max_features', fontweight='bold')
axes[3].set_xlabel('max_features'); axes[3].set_ylabel('Accuracy')
axes[3].legend(); axes[3].set_ylim(0.5, 1.05)
axes[3].set_xticks(mf_vals)

plt.suptitle('Hyperparameter Effects on Train vs Test Accuracy (Wine Dataset)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Decision Boundary Visualisation via PCA (2D)
# ─────────────────────────────────────────────────────────────

# Reduce to 2D with PCA
pca = PCA(n_components=2, random_state=42)
X_pca_train = pca.fit_transform(X_train)
X_pca_test  = pca.transform(X_test)

# Train decision tree on 2D PCA features
clf_2d = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_2d.fit(X_pca_train, y_train)

acc_2d = accuracy_score(y_test, clf_2d.predict(X_pca_test))
print(f"2D PCA tree accuracy: {acc_2d:.4f}  (information loss from PCA is expected)")

# Build a meshgrid for the decision boundary
x_min, x_max = X_pca_train[:, 0].min() - 1, X_pca_train[:, 0].max() + 1
y_min, y_max = X_pca_train[:, 1].min() - 1, X_pca_train[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400),
                     np.linspace(y_min, y_max, 400))

Z = clf_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
cmap_bg   = plt.cm.Pastel1
cmap_pts  = plt.cm.Set1

# Filled contour (background)
ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg, levels=[-0.5, 0.5, 1.5, 2.5])
# Decision boundaries
ax.contour(xx, yy, Z, colors='black', linewidths=0.8, levels=[-0.5, 0.5, 1.5, 2.5])

# Test points
scatter_colors = ['#4C72B0', '#DD8452', '#55A868']
markers = ['o', 's', '^']
for cls in range(3):
    mask = y_test == cls
    ax.scatter(
        X_pca_test[mask, 0], X_pca_test[mask, 1],
        c=scatter_colors[cls], marker=markers[cls],
        edgecolors='white', linewidths=0.8,
        s=70, label=f'Class {cls} ({class_names[cls]})', zorder=3
    )

ax.set_xlabel(f'PC1 (explains {pca.explained_variance_ratio_[0]:.1%} of variance)', fontsize=11)
ax.set_ylabel(f'PC2 (explains {pca.explained_variance_ratio_[1]:.1%} of variance)', fontsize=11)
ax.set_title(
    'Decision Boundary of a Decision Tree in 2D PCA Space\n'
    'Note: axis-aligned rectangular regions — characteristic of trees!',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=10, loc='lower right')
plt.tight_layout()
plt.show()

print("\nKey insight: Decision trees partition the feature space into")
print("axis-aligned rectangles (orthogonal hyperplanes).")
print("This is their defining geometric property — very different from")
print("the diagonal boundaries produced by logistic regression or SVMs.")

## Theory: Decision Trees for Regression

The CART algorithm is not limited to classification — it also handles **regression** tasks.

### Key Differences from Classification Trees

| Aspect | Classification Tree | Regression Tree |
|--------|--------------------|-----------------|
| **Target** | Discrete class labels | Continuous numerical values |
| **Leaf prediction** | Majority class | Mean of samples in the leaf |
| **Impurity measure** | Gini / Entropy | MSE (or MAE) |
| **Split criterion** | Max information gain | Min MSE after split |

### Variance Reduction Criterion

The split criterion for regression trees is the **reduction in Mean Squared Error (MSE)**:

$$\text{MSE}(t) = \frac{1}{n_t} \sum_{i \in t} (y_i - \bar{y}_t)^2$$

where $\bar{y}_t$ is the mean of the target values at node $t$. The best split minimises:

$$\frac{n_L}{n} \cdot \text{MSE}(t_L) + \frac{n_R}{n} \cdot \text{MSE}(t_R)$$

### Leaf Prediction

Each leaf node stores the **average target value** of all training samples that fell into it:

$$\hat{y}(x) = \bar{y}_\text{leaf}(x) = \frac{1}{|\mathcal{L}(x)|} \sum_{i \in \mathcal{L}(x)} y_i$$

This means regression trees produce a **piecewise constant** function — the prediction is flat within each rectangular region and jumps at boundaries. For smoother predictions, ensemble methods (Random Forests, Gradient Boosted Trees) are typically preferred.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Regression Tree — Quick Demo
# ─────────────────────────────────────────────────────────────

from sklearn.datasets import fetch_california_housing
from sklearn.metrics import r2_score, mean_squared_error

# Load California Housing dataset
try:
    housing = fetch_california_housing()
    X_reg, y_reg = housing.data, housing.target
    dataset_name = 'California Housing'
except Exception:
    # Fallback to synthetic data
    X_reg, y_reg = make_regression(n_samples=500, n_features=8,
                                    noise=30, random_state=42)
    dataset_name = 'Synthetic Regression'

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Train regression tree
reg_tree = DecisionTreeRegressor(max_depth=4, random_state=42)
reg_tree.fit(X_reg_train, y_reg_train)
y_reg_pred = reg_tree.predict(X_reg_test)

r2  = r2_score(y_reg_test, y_reg_pred)
mse = mean_squared_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mse)

print(f"Regression Tree Demo  ({dataset_name})")
print("=" * 45)
print(f"Dataset shape : {X_reg.shape}")
print(f"max_depth     : 4")
print(f"R² score      : {r2:.4f}")
print(f"RMSE          : {rmse:.4f}")
print(f"Tree leaves   : {reg_tree.get_n_leaves()}")

# Plot predicted vs actual
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: predicted vs actual
axes[0].scatter(y_reg_test, y_reg_pred, alpha=0.3, s=20, color='steelblue', edgecolors='none')
lims = [min(y_reg_test.min(), y_reg_pred.min()),
        max(y_reg_test.max(), y_reg_pred.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual', fontsize=11)
axes[0].set_ylabel('Predicted', fontsize=11)
axes[0].set_title(f'Predicted vs Actual  (R² = {r2:.3f})', fontsize=12, fontweight='bold')
axes[0].legend()

# Residuals
residuals = y_reg_test - y_reg_pred
axes[1].hist(residuals, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='black', ls='--', lw=1.5)
axes[1].set_xlabel('Residual (actual − predicted)', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title(f'Residual Distribution  (RMSE = {rmse:.3f})', fontsize=12, fontweight='bold')

plt.suptitle(f'Decision Tree Regressor — {dataset_name}', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nNote: Regression trees produce piecewise-constant predictions.")
print("The 'step-function' nature is visible in clusters of identical predictions.")

## Exercises

These exercises are designed to deepen your understanding of decision trees beyond what was covered in the workshop. Tackle them in order — each one builds on the previous concepts.

---

### Exercise 1: Cost-Complexity Pruning (`ccp_alpha`)

Scikit-learn's `DecisionTreeClassifier` supports **post-pruning** via the `ccp_alpha` parameter (cost-complexity parameter). The higher `ccp_alpha`, the more aggressively the tree is pruned.

**Your task:**

1. Use `clf.cost_complexity_pruning_path(X_train, y_train)` to obtain the list of effective `ccp_alpha` values and corresponding impurities.
2. Train one `DecisionTreeClassifier` for each `ccp_alpha` in that list.
3. Plot train accuracy and test accuracy as a function of `ccp_alpha`.
4. Find the `ccp_alpha` that maximises test accuracy and compare the size of that tree to the unpruned tree.

```python
# Hint
path = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
# ... train a tree for each alpha ...
```

**Expected outcome**: You should observe that as `ccp_alpha` increases, the tree shrinks (fewer leaves), training accuracy decreases, but test accuracy first improves then degrades.

---

### Exercise 2: Implement a 1-Feature Split Finder from Scratch

In this exercise you will implement the core of the CART algorithm: finding the best threshold for a single feature.

**Your task:**

1. Write a function `best_split(X_feature, y)` that:
   - Iterates over all unique values of `X_feature` as candidate thresholds
   - For each threshold, computes the information gain using your `information_gain` function from Cell 7
   - Returns the threshold that maximises information gain and the corresponding IG value
2. Apply your function to **each feature** of the Wine dataset training set
3. Identify which feature + threshold provides the highest information gain
4. Compare your result with the root node of the tree trained in Cell 11

```python
def best_split(X_feature, y):
    best_threshold = None
    best_ig = -np.inf
    for threshold in np.unique(X_feature):
        left_mask  = X_feature <= threshold
        right_mask = ~left_mask
        ig = information_gain(y, y[left_mask], y[right_mask], criterion='gini')
        if ig > best_ig:
            best_ig, best_threshold = ig, threshold
    return best_threshold, best_ig
```

**Expected outcome**: Your best split should match the root node of the shallow tree from Cell 11.

---

### Exercise 3: One-vs-Rest Strategy for Multiclass Classification

The Wine dataset has 3 classes (0, 1, 2). Instead of training a single multiclass tree, we can train **one binary tree per class** (One-vs-Rest / OvR).

**Your task:**

1. For each class $k \in \{0, 1, 2\}$, create a binary label vector `y_binary` where:
   - `y_binary[i] = 1` if `y[i] == k`, else `y_binary[i] = 0`
2. Train a `DecisionTreeClassifier(max_depth=4)` on each binary problem
3. For prediction, use the predicted **probability** from each OvR tree (`predict_proba`), and assign the class with the highest probability
4. Compare the accuracy of your OvR approach with the single multiclass tree from Cell 11
5. *(Bonus)*: Visualise the 3 binary trees side-by-side

**Expected outcome**: The OvR approach may have slightly different accuracy. Think about why: does having 3 separate simpler problems help or hurt compared to one joint optimisation?

## Summary

### Key Takeaways

In this workshop, we explored **Decision Trees** — one of the most interpretable and versatile algorithms in supervised machine learning.

**Core concepts covered:**

1. **How trees are built**: Recursive binary splitting using CART, guided by impurity measures (Gini or Entropy). At each node, we find the feature + threshold pair that maximises information gain.

2. **Impurity measures**: Gini impurity and entropy both measure class mixing in a node. Information Gain quantifies how much a split reduces impurity. Both measures yield similar results in practice.

3. **Interpretability**: The tree structure can be visualised and read as a set of IF-THEN rules. Feature importances tell us which variables drive the model's decisions most.

4. **Overfitting control**: Unlimited depth leads to perfect memorisation of training data (0 training error, poor generalisation). Key tools: `max_depth`, `min_samples_leaf`, `min_samples_split`, and `ccp_alpha` (cost-complexity pruning).

5. **Decision boundaries**: Trees create axis-aligned, rectangular partitions of the feature space — a distinctive and interpretable geometric pattern.

6. **Regression trees**: CART extends naturally to regression by replacing entropy/Gini with MSE and predicting the mean of leaf samples.

---

### Pros and Cons of Decision Trees

| Pros | Cons |
|------|------|
| Highly interpretable (white-box model) | Prone to overfitting (high variance) |
| Handles mixed feature types natively | Instability: small data changes → very different trees |
| No feature scaling required | Axis-aligned boundaries can struggle with diagonal patterns |
| Fast to train and predict | Biased feature importances (MDI) |
| Can model non-linear relationships | Not competitive with ensembles on complex tasks |
| Natural handling of multi-class problems | |

---

### What's Next: Workshop 11 — KNN & Naive Bayes

In the next workshop we will explore two very different approaches to classification:

- **K-Nearest Neighbours (KNN)**: A non-parametric, instance-based learner that classifies by majority vote among the $k$ closest training points. No model is fitted — the training data *is* the model.

- **Naive Bayes**: A probabilistic classifier based on Bayes' theorem with a strong ("naive") conditional independence assumption between features. Extremely fast, works well with text and high-dimensional sparse data.

Both algorithms will be contrasted with decision trees in terms of interpretability, computational cost, and generalisation behaviour.

---

*Workshop 10 — Decision Trees | Machine Learning Course*